# 02 — Download SDO/HMI SHARP magnetograms

Retrieves the Space-weather HMI Active Region Patch (SHARP) CEA vector magnetograms used to produce **Figures 8 and 9** and the SHARP indices summarised in **Tables 3 and 4**.

| Active region | HARPNUM | Dates plotted in the paper |
|---|---|---|
| AR 13663 | 11142 | 2024-05-03, 2024-05-05, 2024-05-06 |
| AR 13664 | 11149 | 2024-05-08, 2024-05-09, 2024-05-10 |

> **Note.** The full SHARP cubes are several hundred MB and are *not* stored in this repository. Run this notebook once to populate `data/sharp/` before running notebook `08`.

Two acquisition routes are shown below:
1. **`sunpy.net.Fido`** — simplest, no JSOC registration required.
2. **`drms`** — the official JSOC Python client; needed if you want to export full SHARP series or non-standard segments.


In [ ]:
from pathlib import Path
import astropy.units as u
from sunpy.net import Fido, attrs as a

SHARP_DIR = Path('..').resolve() / 'data' / 'sharp'
SHARP_DIR.mkdir(parents=True, exist_ok=True)

# Segments needed for the Bz / Bh paper figures
SEGMENTS = ('Br', 'Bt', 'Bp')


## Option 1 — `sunpy.net.Fido`

Single-day example for AR 13663 (HARPNUM 11142) on 2024-05-03 at 00:00 TAI.


In [ ]:
harpnum = 11142            # AR 13663
date = '2024-05-03'

query = Fido.search(
    a.Time(f'{date}T00:00:00', f'{date}T00:00:30'),
    a.jsoc.Series('hmi.sharp_cea_720s'),
    a.jsoc.PrimeKey('HARPNUM', harpnum),
    a.jsoc.Segment('Br') & a.jsoc.Segment('Bt') & a.jsoc.Segment('Bp'),
    a.jsoc.Notify('YOUR_REGISTERED_EMAIL@example.com'),  # required by JSOC
)
query


In [ ]:
files = Fido.fetch(query, path=str(SHARP_DIR / '{file}'))
files


## Option 2 — `drms` client (bulk download)

Use this when you need a full time series (e.g. one cadence per day for all 15 days of May 2024).


In [ ]:
import drms

client = drms.Client(email='YOUR_REGISTERED_EMAIL@example.com', verbose=True)

# AR 13663 — daily 00:00 TAI from May 1 to May 10
query_str = 'hmi.sharp_cea_720s[11142][2024.05.01_TAI/10d@1d]{Br,Bt,Bp}'
request = client.export(query_str, method='url', protocol='fits')
request.wait()
request.download(str(SHARP_DIR))


In [ ]:
# AR 13664 — daily 00:00 TAI from May 1 to May 15
query_str = 'hmi.sharp_cea_720s[11149][2024.05.01_TAI/15d@1d]{Br,Bt,Bp}'
request = client.export(query_str, method='url', protocol='fits')
request.wait()
request.download(str(SHARP_DIR))
